# Thesis core results

This notebook produces the core tables, plots, and headline values for the bachelor thesis results chapter. It uses the existing simulation outputs in `outputs/csv/` and writes thesis-specific artefacts to `outputs/thesis_figures/` and `outputs/thesis_tables/`.

The analysis focuses on revenue-neutral labour-income tax progressivity, transfer-policy design, capital-return heterogeneity, wealth inequality, disposable-income inequality, and mobility.

## 1. Setup

In [ ]:
import os
from pathlib import Path
import tempfile

import numpy as np
import pandas as pd

MPL_CONFIG_DIR = Path(tempfile.gettempdir()) / "abm_matplotlib"
MPL_CONFIG_DIR.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(MPL_CONFIG_DIR))
XDG_CACHE_DIR = Path(tempfile.gettempdir()) / "abm_cache"
XDG_CACHE_DIR.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("XDG_CACHE_HOME", str(XDG_CACHE_DIR))

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

CSV_DIR = ROOT / "outputs" / "csv"
FIGURE_DIR = ROOT / "outputs" / "thesis_figures"
TABLE_DIR = ROOT / "outputs" / "thesis_tables"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    "figure.figsize": (8, 5),
    "axes.grid": True,
    "grid.alpha": 0.25,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.titlesize": 13,
    "axes.labelsize": 11,
    "legend.frameon": False,
})

def load_csv(name):
    path = CSV_DIR / name
    if path.exists():
        return add_mean_aliases(pd.read_csv(path))
    print(f"Missing optional file: {path}")
    return pd.DataFrame()

def add_mean_aliases(df):
    df = df.copy()
    for column in list(df.columns):
        if column.endswith("_mean"):
            base = column[:-5]
            if base not in df.columns:
                df[base] = df[column]
    return df

final_summary = load_csv("final_summary.csv")
mobility_summary = load_csv("mobility_summary.csv")
scenario_comparison = load_csv("scenario_comparison.csv")
return_heterogeneity_summary = load_csv("return_heterogeneity_summary.csv")
transfer_policy_final_summary = load_csv("transfer_policy_final_summary.csv")
transfer_policy_mobility_summary = load_csv("transfer_policy_mobility_summary.csv")
transfer_policy_decomposition_summary = load_csv("transfer_policy_decomposition_summary.csv")
transfer_policy_yearly_results = load_csv("transfer_policy_yearly_results.csv")
decomposition_summary = load_csv("decomposition_summary.csv")

print("Loaded CSV files from", CSV_DIR)
for name, df in {
    "final_summary": final_summary,
    "mobility_summary": mobility_summary,
    "scenario_comparison": scenario_comparison,
    "return_heterogeneity_summary": return_heterogeneity_summary,
    "transfer_policy_final_summary": transfer_policy_final_summary,
    "transfer_policy_mobility_summary": transfer_policy_mobility_summary,
    "transfer_policy_decomposition_summary": transfer_policy_decomposition_summary,
    "transfer_policy_yearly_results": transfer_policy_yearly_results,
    "decomposition_summary": decomposition_summary,
}.items():
    print(f"{name}: {df.shape}")

## 2. Helper functions

In [ ]:
SCENARIO_ORDER = [
    "flat",
    "swedish_low_progressivity",
    "swedish_baseline_progressivity",
    "swedish_high_progressivity",
]

RETURN_ORDER = [
    "low_return_heterogeneity",
    "baseline_return_heterogeneity",
    "high_return_heterogeneity",
]

TRANSFER_ORDER = [
    "no_transfers",
    "lump_sum_only",
    "universal_plus_safety_floor",
]

def clean_scenario_label(value):
    labels = {
        "flat": "Flat tax",
        "swedish_low_progressivity": "Low progressivity",
        "swedish_baseline_progressivity": "Baseline progressivity",
        "swedish_high_progressivity": "High progressivity",
    }
    return labels.get(value, str(value).replace("_", " ").title())

def clean_return_label(value):
    labels = {
        "low_return_heterogeneity": "Low return heterogeneity",
        "baseline_return_heterogeneity": "Baseline return heterogeneity",
        "high_return_heterogeneity": "High return heterogeneity",
    }
    return labels.get(value, str(value).replace("_", " ").title())

def clean_transfer_label(value):
    labels = {
        "no_transfers": "No transfers",
        "lump_sum_only": "Lump-sum only",
        "universal_plus_safety_floor": "Universal + safety floor",
    }
    return labels.get(value, str(value).replace("_", " ").title())

def pp_diff(value, baseline):
    return 100 * (value - baseline)

def require_columns(df, columns, name="dataframe"):
    missing = [column for column in columns if column not in df.columns]
    if missing:
        print(f"Missing columns in {name}: {missing}")
        print(f"Available columns in {name}: {list(df.columns)}")
        return False
    return True

def table_formatter(df, columns, rename=None, decimals=3):
    if df.empty:
        return df
    output = df[columns].copy()
    if rename:
        output = output.rename(columns=rename)
    numeric_columns = output.select_dtypes(include=["number"]).columns
    output[numeric_columns] = output[numeric_columns].round(decimals)
    return output

def scenario_column(df):
    if "tax_scenario" in df.columns:
        return "tax_scenario"
    if "scenario" in df.columns:
        return "scenario"
    return None

def baseline_panel(df):
    if df.empty:
        return df
    panel = df.copy()
    sc_col = scenario_column(panel)
    if sc_col and sc_col != "tax_scenario":
        panel = panel.rename(columns={sc_col: "tax_scenario"})
    if "return_preset" in panel.columns:
        panel = panel[panel["return_preset"] == "baseline_return_heterogeneity"]
    if "transfer_policy" in panel.columns:
        panel = panel[panel["transfer_policy"] == "universal_plus_safety_floor"]
    return panel

## 3. Main baseline results

The baseline comparison uses `baseline_return_heterogeneity` and the default transfer-policy setting `universal_plus_safety_floor` when those dimensions are available.

In [ ]:
main_source = transfer_policy_final_summary if not transfer_policy_final_summary.empty else final_summary
main_panel = baseline_panel(main_source)

main_columns = [
    "tax_scenario",
    "final_gini",
    "final_disposable_income_gini",
    "final_top_10_share",
    "final_top_1_share",
    "shorrocks_index",
    "top_20_persistence",
    "final_capital_income_share",
]

if require_columns(main_panel, main_columns, "main_panel"):
    main_table = main_panel[main_columns].copy()
    main_table["scenario_label"] = main_table["tax_scenario"].map(clean_scenario_label)
    main_table["scenario_order"] = main_table["tax_scenario"].map({name: i for i, name in enumerate(SCENARIO_ORDER)})
    main_table = main_table.sort_values("scenario_order")
    flat_row = main_table.loc[main_table["tax_scenario"] == "flat"].iloc[0]
    for column in ["final_gini", "final_disposable_income_gini", "final_top_10_share", "final_top_1_share", "shorrocks_index", "top_20_persistence", "final_capital_income_share"]:
        main_table[f"{column}_diff_vs_flat_pp"] = pp_diff(main_table[column], flat_row[column])
    display(table_formatter(
        main_table,
        ["scenario_label"] + [column for column in main_table.columns if column.startswith("final_") or column in ["shorrocks_index", "top_20_persistence"]],
        {"scenario_label": "Scenario"},
    ))
else:
    main_table = pd.DataFrame()

## 4. Core result plots

In [ ]:
plot_panel = baseline_panel(transfer_policy_yearly_results)
if plot_panel.empty:
    plot_panel = baseline_panel(scenario_comparison)

def plot_time_series(df, metric, ylabel, filename):
    required = ["tax_scenario", "year", metric]
    if not require_columns(df, required, filename):
        return
    fig, ax = plt.subplots(figsize=(8, 5))
    for scenario in SCENARIO_ORDER:
        data = df[df["tax_scenario"] == scenario]
        if data.empty:
            continue
        ax.plot(data["year"], data[metric], marker="o", linewidth=2, label=clean_scenario_label(scenario))
    ax.set_xlabel("Year")
    ax.set_ylabel(ylabel)
    ax.legend()
    fig.tight_layout()
    path = FIGURE_DIR / filename
    fig.savefig(path, dpi=200)
    display(fig)
    print("Saved", path)

if "scenario" in plot_panel.columns and "tax_scenario" not in plot_panel.columns:
    plot_panel = plot_panel.rename(columns={"scenario": "tax_scenario"})

plot_time_series(plot_panel, "wealth_gini", "Wealth Gini", "fig_wealth_gini_by_tax_scenario.png")
plot_time_series(plot_panel, "disposable_income_gini", "Disposable-income Gini", "fig_disposable_income_gini_by_tax_scenario.png")
plot_time_series(plot_panel, "top_10_share", "Top 10% wealth share", "fig_top_10_share_by_tax_scenario.png")
plot_time_series(plot_panel, "top_1_share", "Top 1% wealth share", "fig_top_1_share_by_tax_scenario.png")

In [ ]:
# Final Lorenz curves are produced by the simulation runner. If the PNG exists, record its path for the thesis workflow.
lorenz_candidates = [
    ROOT / "outputs" / "figures" / "final_lorenz_curves.png",
    FIGURE_DIR / "final_lorenz_curves.png",
]
for path in lorenz_candidates:
    if path.exists():
        print("Final Lorenz curve figure available at", path)
        break
else:
    print("No final Lorenz curve image found. Re-run scripts/run_scenarios.py to generate it.")

## 5. Income versus wealth inequality comparison

In [ ]:
income_wealth_columns = ["tax_scenario", "final_disposable_income_gini", "final_gini", "final_top_1_share"]
if require_columns(main_panel, income_wealth_columns, "income wealth panel"):
    flat = main_panel.loc[main_panel["tax_scenario"] == "flat"].iloc[0]
    high = main_panel.loc[main_panel["tax_scenario"] == "swedish_high_progressivity"].iloc[0]
    income_vs_wealth = pd.DataFrame([
        {"Measure": "Disposable-income Gini", "Flat": flat["final_disposable_income_gini"], "High progressivity": high["final_disposable_income_gini"], "Change pp": pp_diff(high["final_disposable_income_gini"], flat["final_disposable_income_gini"])},
        {"Measure": "Wealth Gini", "Flat": flat["final_gini"], "High progressivity": high["final_gini"], "Change pp": pp_diff(high["final_gini"], flat["final_gini"])},
        {"Measure": "Top 1% share", "Flat": flat["final_top_1_share"], "High progressivity": high["final_top_1_share"], "Change pp": pp_diff(high["final_top_1_share"], flat["final_top_1_share"])},
    ])
    display(table_formatter(income_vs_wealth, income_vs_wealth.columns.tolist()))
    print("Progressivity reduces disposable-income inequality more strongly than wealth inequality.")
else:
    income_vs_wealth = pd.DataFrame()

## 6. Return heterogeneity robustness

In [ ]:
return_source = transfer_policy_final_summary.copy()
if not return_source.empty:
    return_source = return_source[return_source["transfer_policy"] == "universal_plus_safety_floor"]

return_rows = []
required = ["return_preset", "tax_scenario", "final_gini", "final_top_1_share", "final_disposable_income_gini"]
if require_columns(return_source, required, "return robustness source"):
    for preset in RETURN_ORDER:
        data = return_source[return_source["return_preset"] == preset]
        if data.empty:
            continue
        flat = data.loc[data["tax_scenario"] == "flat"].iloc[0]
        high = data.loc[data["tax_scenario"] == "swedish_high_progressivity"].iloc[0]
        return_rows.append({
            "Return preset": clean_return_label(preset),
            "Flat wealth Gini": flat["final_gini"],
            "High-progressivity wealth Gini": high["final_gini"],
            "Wealth Gini difference pp": pp_diff(high["final_gini"], flat["final_gini"]),
            "Flat top 1% share": flat["final_top_1_share"],
            "High-progressivity top 1% share": high["final_top_1_share"],
            "Top 1% difference pp": pp_diff(high["final_top_1_share"], flat["final_top_1_share"]),
            "Flat disposable-income Gini": flat["final_disposable_income_gini"],
            "High-progressivity disposable-income Gini": high["final_disposable_income_gini"],
            "Disposable-income Gini difference pp": pp_diff(high["final_disposable_income_gini"], flat["final_disposable_income_gini"]),
        })
    return_robustness = pd.DataFrame(return_rows)
    display(table_formatter(return_robustness, return_robustness.columns.tolist()))
else:
    return_robustness = pd.DataFrame()

In [ ]:
def plot_return_metric(df, metric, ylabel, filename):
    required = ["return_preset", "tax_scenario", metric]
    if not require_columns(df, required, filename):
        return
    fig, ax = plt.subplots(figsize=(8, 5))
    x = np.arange(len(RETURN_ORDER))
    width = 0.2
    for i, scenario in enumerate(SCENARIO_ORDER):
        values = []
        for preset in RETURN_ORDER:
            row = df[(df["return_preset"] == preset) & (df["tax_scenario"] == scenario)]
            values.append(row.iloc[0][metric] if not row.empty else np.nan)
        ax.bar(x + (i - 1.5) * width, values, width, label=clean_scenario_label(scenario))
    ax.set_xticks(x)
    ax.set_xticklabels([clean_return_label(p) for p in RETURN_ORDER], rotation=20, ha="right")
    ax.set_ylabel(ylabel)
    ax.legend()
    fig.tight_layout()
    path = FIGURE_DIR / filename
    fig.savefig(path, dpi=200)
    display(fig)
    print("Saved", path)

plot_return_metric(return_source, "final_gini", "Final wealth Gini", "fig_wealth_gini_by_return_preset.png")
plot_return_metric(return_source, "final_top_1_share", "Final top 1% share", "fig_top_1_by_return_preset.png")

if not return_robustness.empty:
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.bar(return_robustness["Return preset"], return_robustness["Wealth Gini difference pp"])
    ax.set_ylabel("High progressivity minus flat, pp")
    ax.set_title("Effect of high progressivity on wealth Gini")
    ax.tick_params(axis="x", rotation=20)
    fig.tight_layout()
    path = FIGURE_DIR / "fig_high_progressivity_effect_by_return_preset.png"
    fig.savefig(path, dpi=200)
    display(fig)
    print("Saved", path)

## 7. Transfer-policy decomposition

In [ ]:
transfer_source = transfer_policy_final_summary.copy()
if not transfer_source.empty:
    transfer_source = transfer_source[transfer_source["return_preset"] == "baseline_return_heterogeneity"]

transfer_rows = []
required = ["transfer_policy", "tax_scenario", "final_gini", "final_disposable_income_gini", "final_top_1_share", "shorrocks_index"]
if require_columns(transfer_source, required, "transfer policy source"):
    for policy in TRANSFER_ORDER:
        data = transfer_source[transfer_source["transfer_policy"] == policy]
        if data.empty:
            continue
        for scenario in ["flat", "swedish_high_progressivity"]:
            row = data.loc[data["tax_scenario"] == scenario].iloc[0]
            transfer_rows.append({
                "Transfer policy": clean_transfer_label(policy),
                "Scenario": clean_scenario_label(scenario),
                "Wealth Gini": row["final_gini"],
                "Disposable-income Gini": row["final_disposable_income_gini"],
                "Top 1% share": row["final_top_1_share"],
                "Shorrocks index": row["shorrocks_index"],
            })
    transfer_decomposition = pd.DataFrame(transfer_rows)
    display(table_formatter(transfer_decomposition, transfer_decomposition.columns.tolist()))
else:
    transfer_decomposition = pd.DataFrame()

In [ ]:
def plot_transfer_metric(df, metric, ylabel, filename):
    required = ["transfer_policy", "tax_scenario", metric]
    if not require_columns(df, required, filename):
        return
    fig, ax = plt.subplots(figsize=(8, 5))
    x = np.arange(len(TRANSFER_ORDER))
    width = 0.2
    for i, scenario in enumerate(SCENARIO_ORDER):
        values = []
        for policy in TRANSFER_ORDER:
            row = df[(df["transfer_policy"] == policy) & (df["tax_scenario"] == scenario)]
            values.append(row.iloc[0][metric] if not row.empty else np.nan)
        ax.bar(x + (i - 1.5) * width, values, width, label=clean_scenario_label(scenario))
    ax.set_xticks(x)
    ax.set_xticklabels([clean_transfer_label(p) for p in TRANSFER_ORDER], rotation=20, ha="right")
    ax.set_ylabel(ylabel)
    ax.legend()
    fig.tight_layout()
    path = FIGURE_DIR / filename
    fig.savefig(path, dpi=200)
    display(fig)
    print("Saved", path)

plot_transfer_metric(transfer_source, "final_gini", "Final wealth Gini", "fig_transfer_policy_wealth_gini.png")
plot_transfer_metric(transfer_source, "final_disposable_income_gini", "Final disposable-income Gini", "fig_transfer_policy_disposable_income_gini.png")
plot_transfer_metric(transfer_source, "final_top_1_share", "Final top 1% share", "fig_transfer_policy_top_1_share.png")

## 8. Mobility results

In [ ]:
mobility_source = transfer_policy_mobility_summary.copy()
if not mobility_source.empty:
    mobility_source = mobility_source[
        (mobility_source["return_preset"] == "baseline_return_heterogeneity")
        & (mobility_source["transfer_policy"] == "universal_plus_safety_floor")
    ]
elif not mobility_summary.empty:
    mobility_source = baseline_panel(mobility_summary)

if "scenario" in mobility_source.columns and "tax_scenario" not in mobility_source.columns:
    mobility_source = mobility_source.rename(columns={"scenario": "tax_scenario"})

mobility_columns = ["tax_scenario", "shorrocks_index", "top_20_persistence", "bottom_40_to_top_40", "rank_correlation"]
if require_columns(mobility_source, mobility_columns, "mobility source"):
    mobility_table = mobility_source[mobility_columns].copy()
    mobility_table["Scenario"] = mobility_table["tax_scenario"].map(clean_scenario_label)
    mobility_table["scenario_order"] = mobility_table["tax_scenario"].map({name: i for i, name in enumerate(SCENARIO_ORDER)})
    mobility_table = mobility_table.sort_values("scenario_order")
    display(table_formatter(mobility_table, ["Scenario", "shorrocks_index", "top_20_persistence", "bottom_40_to_top_40", "rank_correlation"]))
else:
    mobility_table = pd.DataFrame()

In [ ]:
if not mobility_table.empty:
    for metric, ylabel, filename in [
        ("shorrocks_index", "Shorrocks mobility index", "fig_shorrocks_by_tax_scenario.png"),
        ("top_20_persistence", "Top 20% persistence", "fig_top_20_persistence_by_tax_scenario.png"),
    ]:
        fig, ax = plt.subplots(figsize=(7, 4))
        ax.bar(mobility_table["Scenario"], mobility_table[metric])
        ax.set_ylabel(ylabel)
        ax.tick_params(axis="x", rotation=20)
        fig.tight_layout()
        path = FIGURE_DIR / filename
        fig.savefig(path, dpi=200)
        display(fig)
        print("Saved", path)

## 9. Key thesis numbers

In [ ]:
if not main_panel.empty and require_columns(main_panel, ["tax_scenario", "final_disposable_income_gini", "final_gini", "final_top_1_share", "shorrocks_index", "top_20_persistence"], "key numbers"):
    flat = main_panel.loc[main_panel["tax_scenario"] == "flat"].iloc[0]
    high = main_panel.loc[main_panel["tax_scenario"] == "swedish_high_progressivity"].iloc[0]
    print("Key thesis numbers, high progressivity relative to flat:")
    print(f"Disposable-income Gini change: {pp_diff(high['final_disposable_income_gini'], flat['final_disposable_income_gini']):.2f} percentage points")
    print(f"Wealth Gini change: {pp_diff(high['final_gini'], flat['final_gini']):.2f} percentage points")
    print(f"Top 1% share change: {pp_diff(high['final_top_1_share'], flat['final_top_1_share']):.2f} percentage points")
    print(f"Shorrocks index change: {pp_diff(high['shorrocks_index'], flat['shorrocks_index']):.2f} percentage points")
    print(f"Top 20% persistence change: {pp_diff(high['top_20_persistence'], flat['top_20_persistence']):.2f} percentage points")

if not transfer_decomposition.empty:
    print("\nEffect of no transfers versus transfer policies:")
    display(table_formatter(transfer_decomposition, transfer_decomposition.columns.tolist()))

if not return_robustness.empty:
    print("\nEffect of high return heterogeneity:")
    display(table_formatter(return_robustness, return_robustness.columns.tolist()))

## 10. Export thesis tables

In [ ]:
exports = {
    "table_main_results.csv": main_table if "main_table" in globals() else pd.DataFrame(),
    "table_income_vs_wealth.csv": income_vs_wealth if "income_vs_wealth" in globals() else pd.DataFrame(),
    "table_return_robustness.csv": return_robustness if "return_robustness" in globals() else pd.DataFrame(),
    "table_transfer_decomposition.csv": transfer_decomposition if "transfer_decomposition" in globals() else pd.DataFrame(),
    "table_mobility.csv": mobility_table if "mobility_table" in globals() else pd.DataFrame(),
}

for filename, df in exports.items():
    path = TABLE_DIR / filename
    df.to_csv(path, index=False)
    print("Saved", path, df.shape)